# 13 — Prepare CopyKAT micro-batches for all canonical cells

This notebook reproduces the archived CopyKAT micro-batch preparation step while
remaining safe on the current Windows / 16-GB-RAM system.

Historical scientific design is preserved:

- 351,427 canonical cells;
- 15,163 chromosome-ordered GENCODE-v49 genes;
- `leiden_scvi_main`;
- clusters 8 and 14 as conservative normal-reference pool;
- 2,000 target cells + 1,000 reference anchors per batch;
- `numpy.random.default_rng(seed=42)`;
- 176 batches;
- CopyKAT settings: `id.type="S"`, `ngene.chr=5`, `win.size=25`,
  `KS.cut=0.1`, `distance="euclidean"`, `n.cores=1`.

The archived notebook loaded the full 351,427 × 15,176 object into RAM. This
version reads the validated notebook-10 chromosome-ordered raw-count H5AD
directly from disk and keeps only one target block plus the 1,000 reference
anchors in memory.

Completed batch exports are validated and skipped on rerun. Matrix Market files
are written directly as `.mtx.gz`; the redundant uncompressed `.mtx` copy is not
kept. Generated R scripts can read the compressed matrix directly.

The canonical repository object can have a different row order from the
archived pre-renaming object. Therefore exact historical anchor *cell IDs* are
not asserted; the historical algorithm, reference pool, anchor size, target
coverage, batch count, and matrix dimensions are hard-checked.

**Windows R bridge compatibility fix:** the generated runner now launches Rscript with a sanitized environment that removes inherited conda/Python PATH and runtime variables. This is required because the active Jupyter conda environment caused reproducible `0xC0000005` Rscript crashes, whereas the sanitized environment passed R 4.6.1, Matrix, and CopyKAT 1.2.5 bridge tests.


**v3 CopyKAT fix:** the generated R scripts explicitly load CopyKAT's bundled `sysdata` objects into a private compatibility environment, and robustly parse Python/pandas `True/False` anchor flags. Each batch now requires exactly 1,000 matched normal anchors.


**v4 CopyKAT compatibility fix:** every non-primitive CopyKAT function is retargeted into the same private environment as the bundled annotation objects. This includes downstream helpers such as `convert.all.bins.hg20()`, which also directly references `full.anno`.


## Archived execution checkpoints

```text
input full-gene object            351427 × 15176
usable chromosome-order genes              15163
reference clusters                         8, 14
reference pool cells                       17449
reference anchors                           1000
target chunk size                           2000
number of batches                            176
last target batch                            1427

gene-position report                      (1, 8)
batch plan                              (176, 10)
input manifest                          (176, 14)
prepare output manifest                    (1, 6)
```

In [ ]:
from pathlib import Path
import os, gc, gzip, hashlib, shutil, warnings, csv, subprocess
import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.io import mmwrite

try:
    from anndata.io import read_elem
except ImportError:
    from anndata._io.specs import read_elem

warnings.filterwarnings("ignore")

def _find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()

METADATA_DIR = PROJECT_DIR / "metadata"
CNV_INPUT_DIR = PROJECT_DIR / "data" / "interim" / "integrated" / "CNV_inputs"
OUT_BASE_DIR = CNV_INPUT_DIR / "CopyKAT_HVG8000_r0_6" / "batchwise_micro_all_cells"
for d in (METADATA_DIR, CNV_INPUT_DIR, OUT_BASE_DIR):
    d.mkdir(parents=True, exist_ok=True)

GENE_ORDER_TSV = METADATA_DIR / "CNV_gene_order_gencode_v49_matched_header_HVG8000_r0_6.tsv"
PUBLISHED_GENEORDERED_H5AD = (
    CNV_INPUT_DIR / "GBM_core_4datasets_CNV_input_HVG8000_r0_6_allcells_gene_ordered_countsX.h5ad"
)

def _default_main10_scratch():
    if os.name == "nt" and Path("E:/").exists():
        return Path(r"E:\glioma-cnv-scratch\main_analysis_10")
    return (PROJECT_DIR / ".scratch" / "main_analysis_10").resolve()

SOURCE_CANDIDATES = []
if os.environ.get("GLIOMA_COPYKAT_GENEORDERED_H5AD"):
    SOURCE_CANDIDATES.append(Path(os.environ["GLIOMA_COPYKAT_GENEORDERED_H5AD"]).expanduser().resolve())
SOURCE_CANDIDATES.extend([
    _default_main10_scratch() / PUBLISHED_GENEORDERED_H5AD.name,
    PUBLISHED_GENEORDERED_H5AD,
])

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError("No notebook-10 gene-ordered raw-count H5AD found:\n" + "\n".join(map(str, paths)))

SOURCE_H5AD = first_existing(SOURCE_CANDIDATES)

def _default_scratch_root():
    if os.environ.get("GLIOMA_MAIN13_SCRATCH_ROOT"):
        return Path(os.environ["GLIOMA_MAIN13_SCRATCH_ROOT"]).expanduser().resolve()
    if os.name == "nt" and Path("E:/").exists():
        return Path(r"E:\glioma-cnv-scratch\main_analysis_13")
    return (PROJECT_DIR / ".scratch" / "main_analysis_13").resolve()

SCRATCH_ROOT = _default_scratch_root()
SCRATCH_EXPORT_DIR = SCRATCH_ROOT / "export"
SCRATCH_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

CLUSTER_KEY = "leiden_scvi_main"
REFERENCE_CLUSTERS = {"8", "14"}
TARGET_CHUNK_SIZE = 2000
REF_ANCHOR_SIZE = 1000
RANDOM_SEED = 42
COPYKAT_N_CORES = 1
BATCH_TIMEOUT_HOURS = 8

EXPECTED_N_CELLS = 351427
EXPECTED_FULLGENE_GENES = 15176
EXPECTED_USABLE_GENES = 15163
EXPECTED_REFERENCE_POOL = 17449
EXPECTED_N_BATCHES = 176
EXPECTED_LAST_TARGET_CELLS = 1427

GENE_REPORT_TSV = METADATA_DIR / "copykat_microbatch_gene_position_report_HVG8000_r0_6.tsv"
BATCH_PLAN_TSV = METADATA_DIR / "copykat_microbatch_batch_plan_HVG8000_r0_6.tsv"
INPUT_MANIFEST_TSV = METADATA_DIR / "copykat_microbatch_input_manifest_HVG8000_r0_6.tsv"
PREPARE_MANIFEST_TSV = METADATA_DIR / "copykat_microbatch_prepare_output_manifest_HVG8000_r0_6.tsv"
REPRO_AUDIT_TSV = METADATA_DIR / "copykat_microbatch_reproducibility_audit_HVG8000_r0_6.tsv"

MASTER_PYTHON = OUT_BASE_DIR / "run_all_copykat_microbatches_HVG8000_r0_6.py"
MASTER_PS1 = OUT_BASE_DIR / "run_all_copykat_microbatches_HVG8000_r0_6.ps1"
MASTER_SH = OUT_BASE_DIR / "run_all_copykat_microbatches_HVG8000_r0_6.sh"

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, sep="\t", index=False)
    os.replace(tmp, path)
    print(f"[SAVED] {path} shape={df.shape}")

def hash_strings(values):
    h = hashlib.sha256()
    for value in values:
        h.update(str(value).encode("utf-8"))
        h.update(b"\0")
    return h.hexdigest()

print("Project:", PROJECT_DIR)
print("Notebook-10 raw-count source:", SOURCE_H5AD)
print("CopyKAT output root:", OUT_BASE_DIR)
print("Scratch:", SCRATCH_ROOT)

In [ ]:
# Metadata-only validation of the notebook-10 chromosome-ordered raw-count source.
if not GENE_ORDER_TSV.exists():
    raise FileNotFoundError(GENE_ORDER_TSV)

with h5py.File(SOURCE_H5AD, "r") as h:
    if "X" not in h or "obs" not in h or "var" not in h:
        raise KeyError("Notebook-10 source lacks X/obs/var.")
    xg = h["X"]
    encoding = xg.attrs.get("encoding-type", "")
    if isinstance(encoding, bytes):
        encoding = encoding.decode("utf-8")
    if encoding != "csr_matrix":
        raise AssertionError(f"Expected CSR X, got {encoding!r}")
    source_shape = tuple(int(v) for v in xg.attrs["shape"])
    source_nnz = int(xg["data"].shape[0])
    source_indptr = np.asarray(xg["indptr"], dtype=np.int64)
    obs = read_elem(h["obs"])
    var = read_elem(h["var"])
    sample = np.asarray(xg["data"][: min(source_nnz, 1_000_000)])

if source_shape != (EXPECTED_N_CELLS, EXPECTED_USABLE_GENES):
    raise AssertionError(f"Unexpected notebook-10 source shape: {source_shape}")
if len(obs) != EXPECTED_N_CELLS or len(var) != EXPECTED_USABLE_GENES:
    raise AssertionError("obs/var dimensions do not match source X.")
if CLUSTER_KEY not in obs.columns:
    raise KeyError(CLUSTER_KEY)
if not obs.index.is_unique or not var.index.is_unique:
    raise AssertionError("Source obs_names/var_names are not unique.")

finite = sample[np.isfinite(sample)]
if finite.size and (np.any(finite < 0) or not np.allclose(finite, np.round(finite))):
    raise AssertionError("Notebook-10 X is not raw non-negative integer-like counts.")

# Validate exact gene order against notebook-10 GENCODE-v49 table.
gene_order = pd.read_csv(GENE_ORDER_TSV, sep="\t")
gene_order.columns = [str(c).strip() for c in gene_order.columns]
gene_col = next((c for c in ["gene", "gene_symbol", "symbol", "Gene"] if c in gene_order.columns), None)
chr_col = next((c for c in ["chr", "chromosome", "cnv_chr"] if c in gene_order.columns), None)
start_col = next((c for c in ["start", "cnv_start"] if c in gene_order.columns), None)
end_col = next((c for c in ["end", "cnv_end"] if c in gene_order.columns), None)
if any(v is None for v in [gene_col, chr_col, start_col, end_col]):
    raise ValueError(f"Cannot identify gene-order columns: {list(gene_order.columns)}")

gene_order = gene_order[[gene_col, chr_col, start_col, end_col]].copy()
gene_order.columns = ["gene", "chr", "start", "end"]
gene_order["gene"] = gene_order["gene"].astype(str)
gene_order["chr"] = gene_order["chr"].astype(str).str.replace("^chr", "", regex=True)
gene_order["start"] = pd.to_numeric(gene_order["start"], errors="coerce")
gene_order["end"] = pd.to_numeric(gene_order["end"], errors="coerce")
gene_order = gene_order.dropna(subset=["gene", "chr", "start", "end"]).drop_duplicates("gene")

chr_order = {str(i): i for i in range(1, 23)}
chr_order.update({"X": 23, "Y": 24, "M": 25, "MT": 25, "mitochondria": 25})
gene_order["chr_order"] = gene_order["chr"].map(chr_order).fillna(99).astype(int)
gene_order = gene_order.sort_values(
    ["chr_order", "start", "end", "gene"], kind="mergesort"
).reset_index(drop=True)

usable_genes = gene_order["gene"].astype(str).to_numpy()
source_genes = var.index.astype(str).to_numpy()
if len(usable_genes) != EXPECTED_USABLE_GENES:
    raise AssertionError(f"Usable-gene count changed: {len(usable_genes)}")

source_gene_index = pd.Index(source_genes)
usable_gene_idx = source_gene_index.get_indexer(usable_genes)

if np.any(usable_gene_idx < 0):
    missing_genes = [
        usable_genes[i]
        for i in np.where(usable_gene_idx < 0)[0][:20]
    ]
    raise RuntimeError(
        "Canonical CopyKAT gene-order TSV contains genes missing from "
        f"notebook-10 source X. Examples: {missing_genes}"
    )

if len(np.unique(usable_gene_idx)) != EXPECTED_USABLE_GENES:
    raise RuntimeError("Gene-order permutation is not one-to-one.")

gene_order_identity = np.array_equal(
    usable_gene_idx,
    np.arange(EXPECTED_USABLE_GENES),
)

print(
    "Notebook-10 X already in historical CopyKAT gene order:",
    gene_order_identity,
)

cluster_values = obs[CLUSTER_KEY].astype(str).to_numpy()
ref_pool = np.where(pd.Series(cluster_values).isin(REFERENCE_CLUSTERS).to_numpy())[0]
if len(ref_pool) != EXPECTED_REFERENCE_POOL:
    raise AssertionError(f"Reference pool changed: {len(ref_pool)} vs {EXPECTED_REFERENCE_POOL}")

gene_report = pd.DataFrame([{
    "input_h5ad": str(SOURCE_H5AD),
    "gene_order_file": str(GENE_ORDER_TSV),
    "counts_source": "notebook10_geneordered_X_raw_counts",
    "n_cells": EXPECTED_N_CELLS,
    "n_genes_total": EXPECTED_FULLGENE_GENES,
    "n_genes_usable": EXPECTED_USABLE_GENES,
    "target_chunk_size": TARGET_CHUNK_SIZE,
    "ref_anchor_size": REF_ANCHOR_SIZE,
}])
if gene_report.shape != (1, 8):
    raise AssertionError("Gene-position report structure changed.")
write_tsv_replace(gene_report, GENE_REPORT_TSV)
display(gene_report)

# Exact historical batching algorithm on the canonical repository row order.
rng = np.random.default_rng(RANDOM_SEED)
all_indices = np.arange(EXPECTED_N_CELLS, dtype=np.int64)
ref_anchor_indices = np.sort(
    rng.choice(ref_pool, size=REF_ANCHOR_SIZE, replace=False)
).astype(np.int64, copy=False)

target_chunks = [
    all_indices[i:i + TARGET_CHUNK_SIZE]
    for i in range(0, EXPECTED_N_CELLS, TARGET_CHUNK_SIZE)
]

rows = []
for i, target_idx in enumerate(target_chunks, start=1):
    batch_id = f"copykat_microbatch_{i:04d}"
    batch_idx = np.unique(np.concatenate([target_idx, ref_anchor_indices]))
    rows.append({
        "batch_id": batch_id,
        "batch_num": i,
        "target_start_index": int(target_idx[0]),
        "target_end_index_exclusive": int(target_idx[-1] + 1),
        "n_target_cells": int(len(target_idx)),
        "n_reference_anchor_cells_requested": REF_ANCHOR_SIZE,
        "n_cells_exported_unique": int(len(batch_idx)),
        "batch_dir": str(OUT_BASE_DIR / batch_id),
        "r_script": str(OUT_BASE_DIR / batch_id / f"run_copykat_microbatch_{i:04d}.R"),
        "expected_prediction": str(OUT_BASE_DIR / batch_id / f"{batch_id}_prediction.tsv"),
    })

batch_plan = pd.DataFrame(rows)
if batch_plan.shape != (176, 10):
    raise AssertionError(f"Historical batch-plan shape changed: {batch_plan.shape}")
if len(batch_plan) != EXPECTED_N_BATCHES:
    raise AssertionError("Historical number of batches changed.")
if int(batch_plan["n_target_cells"].sum()) != EXPECTED_N_CELLS:
    raise AssertionError("Target coverage is not exactly 351,427 cells.")
if int(batch_plan.iloc[-1]["n_target_cells"]) != EXPECTED_LAST_TARGET_CELLS:
    raise AssertionError("Historical last target batch changed.")
if len(np.unique(ref_anchor_indices)) != REF_ANCHOR_SIZE:
    raise AssertionError("Reference anchors are not unique.")
if not set(cluster_values[ref_anchor_indices]).issubset(REFERENCE_CLUSTERS):
    raise AssertionError("An anchor lies outside clusters 8/14.")

write_tsv_replace(batch_plan, BATCH_PLAN_TSV)
display(batch_plan.head())
display(batch_plan.tail())
print("source shape:", source_shape, "nnz:", f"{source_nnz:,}")
print("reference pool:", len(ref_pool))
print("reference anchors:", len(ref_anchor_indices))
print("n batches:", len(batch_plan))

In [ ]:
R_TEMPLATE = """
options(stringsAsFactors = FALSE)
options(timeout = 1000)

suppressPackageStartupMessages({{
  library(Matrix)
}})

if (!requireNamespace("copykat", quietly = TRUE)) {{
  stop("The R package 'copykat' is not installed.")
}}

# ------------------------------------------------------------------
# CopyKAT 1.2.5 bundled-data compatibility
# ------------------------------------------------------------------
copykat_data_env <- new.env(parent = emptyenv())

data_loaded <- FALSE

try({{
  suppressWarnings(
    utils::data(
      "sysdata",
      package = "copykat",
      envir = copykat_data_env
    )
  )
  data_loaded <- length(ls(copykat_data_env, all.names = TRUE)) > 0
}}, silent = TRUE)

if (!data_loaded) {{
  data_dir <- system.file("data", package = "copykat")
  raw_sysdata <- file.path(data_dir, "sysdata.rda")

  if (file.exists(raw_sysdata)) {{
    load(raw_sysdata, envir = copykat_data_env)
    data_loaded <- length(ls(copykat_data_env, all.names = TRUE)) > 0
  }}
}}

required_copykat_data <- c(
  "full.anno",
  "cyclegenes",
  "DNA.hg20"
)

missing_copykat_data <- required_copykat_data[
  !vapply(
    required_copykat_data,
    exists,
    logical(1),
    envir = copykat_data_env,
    inherits = FALSE
  )
]

if (length(missing_copykat_data) > 0) {{
  stop(
    "CopyKAT bundled data could not be loaded: ",
    paste(missing_copykat_data, collapse = ", "),
    ". Objects found: ",
    paste(ls(copykat_data_env, all.names = TRUE), collapse = ", ")
  )
}}

message(
  "[COPYKAT DATA] full.anno rows=",
  nrow(get("full.anno", envir = copykat_data_env)),
  " | cyclegenes=",
  length(get("cyclegenes", envir = copykat_data_env)),
  " | DNA.hg20 rows=",
  nrow(get("DNA.hg20", envir = copykat_data_env))
)

copykat_ns <- asNamespace("copykat")

copykat_compat_env <- new.env(
  parent = copykat_ns
)

for (nm in ls(copykat_data_env, all.names = TRUE)) {{
  assign(
    nm,
    get(nm, envir = copykat_data_env, inherits = FALSE),
    envir = copykat_compat_env
  )
}}

copykat_function_names <- ls(
  copykat_ns,
  all.names = TRUE
)

n_functions_copied <- 0L

for (nm in copykat_function_names) {{
  obj <- get(
    nm,
    envir = copykat_ns,
    inherits = FALSE
  )

  if (
    is.function(obj)
    && !is.primitive(obj)
  ) {{
    fn <- obj
    environment(fn) <- copykat_compat_env

    assign(
      nm,
      fn,
      envir = copykat_compat_env
    )

    n_functions_copied <- n_functions_copied + 1L
  }}
}}

message(
  "[COPYKAT COMPAT] functions retargeted=",
  n_functions_copied
)

required_compat_functions <- c(
  "copykat",
  "annotateGenes.hg20",
  "convert.all.bins.hg20"
)

missing_compat_functions <- required_compat_functions[
  !vapply(
    required_compat_functions,
    exists,
    logical(1),
    envir = copykat_compat_env,
    inherits = FALSE
  )
]

if (length(missing_compat_functions) > 0) {{
  stop(
    "CopyKAT compatibility environment is missing functions: ",
    paste(missing_compat_functions, collapse = ", ")
  )
}}

copykat_compat <- get(
  "copykat",
  envir = copykat_compat_env,
  inherits = FALSE
)

batch_id <- "{batch_id}"
base_dir <- "{batch_dir}"

mtx_gz_file <- file.path(base_dir, "copykat_raw_counts_genes_by_cells.mtx.gz")
mtx_file <- file.path(base_dir, "copykat_raw_counts_genes_by_cells.mtx")
genes_file <- file.path(base_dir, "copykat_genes.tsv")
cells_file <- file.path(base_dir, "copykat_barcodes.tsv")
meta_file <- file.path(base_dir, "copykat_cell_metadata.tsv")

out_prefix <- file.path(base_dir, batch_id)
out_tsv <- paste0(out_prefix, "_prediction.tsv")
out_rds <- paste0(out_prefix, "_prediction.rds")
out_obj <- paste0(out_prefix, "_full_object.rds")
out_info <- paste0(out_prefix, "_run_info.tsv")

message("[BATCH] ", batch_id)

if (file.exists(out_tsv)) {{
  message("[SKIP] prediction exists: ", out_tsv)
  quit(save = "no", status = 0)
}}

genes <- read.delim(genes_file, header = FALSE, stringsAsFactors = FALSE)[[1]]
cells <- read.delim(cells_file, header = FALSE, stringsAsFactors = FALSE)[[1]]
meta <- read.delim(
  meta_file,
  header = TRUE,
  sep = "\t",
  stringsAsFactors = FALSE,
  check.names = FALSE
)

if (file.exists(mtx_file) && file.info(mtx_file)$size > 0) {{
  message("[READ MTX] ", mtx_file)
  raw_sparse <- Matrix::readMM(mtx_file)
}} else {{
  if (!file.exists(mtx_gz_file)) stop("Missing MTX input: ", mtx_gz_file)
  message("[READ MTX.GZ] ", mtx_gz_file)
  con <- gzfile(mtx_gz_file, open = "rt")
  raw_sparse <- Matrix::readMM(con)
  close(con)
}}

rownames(raw_sparse) <- make.unique(as.character(genes))
colnames(raw_sparse) <- make.unique(as.character(cells))

message(
  "[SPARSE] genes x cells: ",
  nrow(raw_sparse),
  " x ",
  ncol(raw_sparse)
)

rawmat <- as.matrix(raw_sparse)
storage.mode(rawmat) <- "numeric"

message(
  "[DENSE GB] ",
  round(as.numeric(object.size(rawmat)) / 1024^3, 3)
)

anchor_flag <- tolower(
  trimws(
    as.character(
      meta$is_reference_anchor
    )
  )
)

is_anchor <- anchor_flag %in% c(
  "true",
  "t",
  "1",
  "yes",
  "y"
)

norm_cells_requested <- as.character(
  meta$cell_id[
    is_anchor
  ]
)

norm_cells <- intersect(
  norm_cells_requested,
  colnames(rawmat)
)

message(
  "[NORM anchors requested] ",
  length(norm_cells_requested)
)

message(
  "[NORM anchors matched] ",
  length(norm_cells)
)

if (length(norm_cells_requested) != 1000) {{
  stop(
    "Expected exactly 1000 reference anchors in metadata, found ",
    length(norm_cells_requested)
  )
}}

if (length(norm_cells) != length(norm_cells_requested)) {{
  stop(
    "Reference-anchor cell IDs do not match the count-matrix column names: ",
    length(norm_cells),
    " / ",
    length(norm_cells_requested),
    " matched."
  )
}}

args <- list(
  rawmat = rawmat,
  id.type = "S",
  ngene.chr = 5,
  win.size = 25,
  KS.cut = 0.1,
  sam.name = batch_id,
  distance = "euclidean",
  genome = "hg20",
  n.cores = {n_cores}
)

formal_names <- names(formals(copykat_compat))

if ("plot.genes" %in% formal_names) {{
  args$plot.genes <- "FALSE"
}}

if (
  "norm.cell.names" %in% formal_names
  && length(norm_cells) > 0
) {{
  args$norm.cell.names <- norm_cells
}}

message("[RUN CopyKAT]")

copykat_result <- do.call(
  copykat_compat,
  args
)

saveRDS(copykat_result, out_obj)

prediction <- NULL

if (
  is.list(copykat_result)
  && "prediction" %in% names(copykat_result)
) {{
  prediction <- copykat_result$prediction
}} else if (is.data.frame(copykat_result)) {{
  prediction <- copykat_result
}}

if (is.null(prediction)) {{
  pred_files <- list.files(
    base_dir,
    pattern = "prediction.*txt$|prediction.*tsv$|copykat.pred",
    full.names = TRUE
  )

  message(
    "[WARNING] prediction object not found. Files: ",
    paste(pred_files, collapse = ", ")
  )

  info <- data.frame(
    batch_id = batch_id,
    status = "no_prediction_object",
    n_genes = nrow(rawmat),
    n_cells = ncol(rawmat)
  )

  write.table(
    info,
    out_info,
    sep = "\t",
    row.names = FALSE,
    quote = FALSE
  )

  quit(save = "no", status = 2)
}}

pred_df <- as.data.frame(
  prediction,
  stringsAsFactors = FALSE
)

if (!("cell.names" %in% colnames(pred_df))) {{
  pred_df <- cbind(
    cell.names = rownames(pred_df),
    pred_df
  )
}}

write.table(
  pred_df,
  out_tsv,
  sep = "\t",
  row.names = FALSE,
  quote = FALSE
)

saveRDS(pred_df, out_rds)

info <- data.frame(
  batch_id = batch_id,
  status = "completed",
  n_genes = nrow(rawmat),
  n_cells = ncol(rawmat),
  n_norm_cells = length(norm_cells),
  n_prediction_rows = nrow(pred_df),
  prediction_tsv = out_tsv
)

write.table(
  info,
  out_info,
  sep = "\t",
  row.names = FALSE,
  quote = FALSE
)

message("[DONE] ", batch_id)
"""


In [ ]:
def contiguous_runs(rows):
    rows = np.asarray(rows, dtype=np.int64)
    if rows.size == 0:
        return []
    out = []
    start = prev = int(rows[0])
    for value in rows[1:]:
        value = int(value)
        if value == prev + 1:
            prev = value
            continue
        out.append((start, prev + 1))
        start = prev = value
    out.append((start, prev + 1))
    return out

def read_csr_contiguous_rows(xg, indptr, start, end, n_cols):
    d0, d1 = int(indptr[start]), int(indptr[end])
    data = np.array(xg["data"][d0:d1], copy=True)
    indices = np.array(xg["indices"][d0:d1], copy=True)
    local_indptr = (indptr[start:end + 1] - d0).astype(np.int64, copy=True)
    return sp.csr_matrix(
        (data, indices, local_indptr),
        shape=(end - start, n_cols),
        copy=False,
    )

def read_csr_selected_rows(xg, indptr, rows, n_cols):
    rows = np.asarray(rows, dtype=np.int64)
    if rows.size == 0:
        return sp.csr_matrix((0, n_cols), dtype=xg["data"].dtype)
    if not np.all(rows[1:] >= rows[:-1]) or len(np.unique(rows)) != len(rows):
        raise ValueError("Selected rows must be sorted and unique.")
    parts = [
        read_csr_contiguous_rows(xg, indptr, start, end, n_cols)
        for start, end in contiguous_runs(rows)
    ]
    return sp.vstack(parts, format="csr")

def mtx_gz_dims(path):
    try:
        with gzip.open(path, "rt", encoding="ascii") as f:
            if not f.readline().startswith("%%MatrixMarket"):
                return None
            for line in f:
                line = line.strip()
                if line and not line.startswith("%"):
                    a, b, c = line.split()[:3]
                    return int(a), int(b), int(c)
    except Exception:
        return None
    return None

def line_count(path):
    with Path(path).open("r", encoding="utf-8", errors="replace") as f:
        return sum(1 for line in f if line.rstrip("\r\n") != "")

def clean_series(vals):
    return pd.Series(vals).astype(str).replace({"nan": "", "None": "", "<NA>": ""}).values

def as_bool_array(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).to_numpy(dtype=bool)
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
        .to_numpy(dtype=bool)
    )

def batch_paths(row):
    batch_dir = Path(row["batch_dir"])
    batch_num = int(row["batch_num"])
    batch_id = str(row["batch_id"])
    return {
        "batch_dir": batch_dir,
        "mtx": batch_dir / "copykat_raw_counts_genes_by_cells.mtx",
        "mtx_gz": batch_dir / "copykat_raw_counts_genes_by_cells.mtx.gz",
        "genes": batch_dir / "copykat_genes.tsv",
        "barcodes": batch_dir / "copykat_barcodes.tsv",
        "metadata": batch_dir / "copykat_cell_metadata.tsv",
        "r_script": batch_dir / f"run_copykat_microbatch_{batch_num:04d}.R",
        "prediction": batch_dir / f"{batch_id}_prediction.tsv",
    }

def batch_export_valid(row):
    p = batch_paths(row)
    expected_cells = int(row["n_cells_exported_unique"])
    required = [p["mtx_gz"], p["genes"], p["barcodes"], p["metadata"], p["r_script"]]
    if not all(x.exists() for x in required):
        return False
    dims = mtx_gz_dims(p["mtx_gz"])
    if dims is None or dims[0] != EXPECTED_USABLE_GENES or dims[1] != expected_cells:
        return False
    if line_count(p["genes"]) != EXPECTED_USABLE_GENES:
        return False
    if line_count(p["barcodes"]) != expected_cells:
        return False
    try:
        meta = pd.read_csv(p["metadata"], sep="\t")
    except Exception:
        return False
    if len(meta) != expected_cells:
        return False
    needed = {"cell_id", "original_index", "is_target_cell", "is_reference_anchor", "batch_id"}
    if not needed.issubset(meta.columns):
        return False
    if int(as_bool_array(meta["is_target_cell"]).sum()) != int(row["n_target_cells"]):
        return False
    if int(as_bool_array(meta["is_reference_anchor"]).sum()) != REF_ANCHOR_SIZE:
        return False
    return True

def atomic_copy(source, destination):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    tmp = destination.with_suffix(destination.suffix + ".publishing")
    if tmp.exists():
        tmp.unlink()
    shutil.copyfile(source, tmp)
    if tmp.stat().st_size != source.stat().st_size:
        raise IOError("Atomic copy size mismatch.")
    os.replace(tmp, destination)

# Load the 1,000 anchor rows once.
with h5py.File(SOURCE_H5AD, "r") as h:
    ref_anchor_matrix = read_csr_selected_rows(
        h["X"], source_indptr, ref_anchor_indices, EXPECTED_USABLE_GENES
    )

if not gene_order_identity:
    ref_anchor_matrix = ref_anchor_matrix[:, usable_gene_idx].tocsr()

if ref_anchor_matrix.shape != (REF_ANCHOR_SIZE, EXPECTED_USABLE_GENES):
    raise AssertionError("Reference-anchor matrix shape changed.")

print("Reference-anchor matrix:", ref_anchor_matrix.shape, "nnz=", f"{ref_anchor_matrix.nnz:,}")

metadata_candidates = [
    CLUSTER_KEY,
    "annotation_marker_based_HVG8000_r0_6",
    "annotation_marker_based",
    "condition",
    "core_dataset",
    "dataset_id",
    "patient_id",
    "sample_id",
    "CNV_role_recommendation",
    "cnv_role_suggestion_HVG8000_r0_6",
    "cnv_role_suggestion",
]

records = []

with h5py.File(SOURCE_H5AD, "r") as h:
    xg = h["X"]

    for _, row in batch_plan.iterrows():
        batch_id = str(row["batch_id"])
        batch_num = int(row["batch_num"])
        p = batch_paths(row)
        p["batch_dir"].mkdir(parents=True, exist_ok=True)

        start = int(row["target_start_index"])
        end = int(row["target_end_index_exclusive"])
        target_idx = np.arange(start, end, dtype=np.int64)
        batch_idx = np.unique(np.concatenate([target_idx, ref_anchor_indices]))
        is_target = np.isin(batch_idx, target_idx)
        is_ref_anchor = np.isin(batch_idx, ref_anchor_indices)

        if len(batch_idx) != int(row["n_cells_exported_unique"]):
            raise RuntimeError(f"{batch_id}: batch union size changed.")

        if batch_export_valid(row):
            print("[SKIP VALIDATED]", batch_id)
        else:
            print("[EXPORT]", batch_id)

            target_matrix = read_csr_contiguous_rows(
                xg, source_indptr, start, end, EXPECTED_USABLE_GENES
            )

            if not gene_order_identity:
                target_matrix = target_matrix[:, usable_gene_idx].tocsr()

            anchor_not_target = ~np.isin(ref_anchor_indices, target_idx)
            combined = sp.vstack(
                [target_matrix, ref_anchor_matrix[anchor_not_target, :]],
                format="csr",
            )
            combined_positions = np.concatenate(
                [target_idx, ref_anchor_indices[anchor_not_target]]
            )
            order = np.argsort(combined_positions, kind="stable")

            if not np.array_equal(combined_positions[order], batch_idx):
                raise RuntimeError(f"{batch_id}: historical union row order reconstruction failed.")

            batch_matrix = combined[order, :].tocsr()
            if batch_matrix.shape != (len(batch_idx), EXPECTED_USABLE_GENES):
                raise AssertionError(f"{batch_id}: matrix shape changed.")

            scratch_gz = SCRATCH_EXPORT_DIR / f"{batch_id}.mtx.gz"
            if scratch_gz.exists():
                scratch_gz.unlink()

            with gzip.open(scratch_gz, "wb", compresslevel=6) as gz:
                mmwrite(gz, batch_matrix.T.tocoo())

            dims = mtx_gz_dims(scratch_gz)
            if dims is None or dims[0] != EXPECTED_USABLE_GENES or dims[1] != len(batch_idx):
                raise RuntimeError(f"{batch_id}: scratch Matrix Market failed validation.")

            atomic_copy(scratch_gz, p["mtx_gz"])
            scratch_gz.unlink(missing_ok=True)

            # Storage-only change: no redundant uncompressed .mtx is retained.
            if p["mtx"].exists():
                p["mtx"].unlink()

            pd.Series(usable_genes).to_csv(p["genes"], sep="\t", index=False, header=False)
            pd.Series(obs.index[batch_idx].astype(str)).to_csv(
                p["barcodes"], sep="\t", index=False, header=False
            )

            meta = pd.DataFrame({
                "cell_id": obs.index[batch_idx].astype(str),
                "original_index": batch_idx.astype(int),
                "is_target_cell": is_target.astype(bool),
                "is_reference_anchor": is_ref_anchor.astype(bool),
                "batch_id": batch_id,
            })
            for col in metadata_candidates:
                if col in obs.columns:
                    meta[col] = clean_series(obs.iloc[batch_idx][col].values)
            meta.to_csv(p["metadata"], sep="\t", index=False)

            # R accepts forward slashes on Windows.
            r_text = R_TEMPLATE.format(
                batch_id=batch_id,
                batch_dir=p["batch_dir"].as_posix(),
                n_cores=COPYKAT_N_CORES,
            )
            p["r_script"].write_text(r_text, encoding="utf-8")

            if not batch_export_valid(row):
                raise RuntimeError(f"{batch_id}: published batch failed validation.")

            del target_matrix, combined, batch_matrix
            gc.collect()

        records.append({
            "batch_id": batch_id,
            "batch_num": batch_num,
            "batch_dir": str(p["batch_dir"]),
            "n_cells_exported": int(len(batch_idx)),
            "n_target_cells": int(is_target.sum()),
            "n_reference_anchor_cells": int(is_ref_anchor.sum()),
            "n_genes": EXPECTED_USABLE_GENES,
            "mtx_gz": str(p["mtx_gz"]),
            "mtx": str(p["mtx"]),
            "genes": str(p["genes"]),
            "barcodes": str(p["barcodes"]),
            "metadata": str(p["metadata"]),
            "r_script": str(p["r_script"]),
            "expected_prediction": str(p["prediction"]),
        })

        # Incremental checkpoint for safe restart.
        write_tsv_replace(pd.DataFrame(records), INPUT_MANIFEST_TSV)

manifest = pd.DataFrame(records)
if manifest.shape != (176, 14):
    raise AssertionError(f"Historical input-manifest shape changed: {manifest.shape}")
if int(manifest["n_target_cells"].sum()) != EXPECTED_N_CELLS:
    raise AssertionError("Exported target-cell coverage changed.")
if not (manifest["n_genes"] == EXPECTED_USABLE_GENES).all():
    raise AssertionError("At least one batch has wrong gene count.")
if not (manifest["n_reference_anchor_cells"] == REF_ANCHOR_SIZE).all():
    raise AssertionError("At least one batch lost reference-anchor labels.")

write_tsv_replace(manifest, INPUT_MANIFEST_TSV)
display(manifest.head())
display(manifest.tail())
print("[ALL 176 COPYKAT INPUT BATCHES VALIDATED]")

In [ ]:
# Generate a cross-platform runner with the historical 8-hour per-batch timeout.
runner = r'''#!/usr/bin/env python
from pathlib import Path
import csv
import os
import re
import shutil
import subprocess
import sys

BASE_DIR = Path(r"__BASE_DIR__")
MANIFEST = Path(r"__MANIFEST__")
TIMEOUT_SECONDS = __TIMEOUT_SECONDS__


def find_rscript():
    candidates = []

    explicit = os.environ.get("RSCRIPT_BIN")
    if explicit:
        candidates.append(Path(explicit))

    which = shutil.which("Rscript")
    if which:
        candidates.append(Path(which))

    if os.name == "nt":
        program_files = Path(
            os.environ.get("ProgramFiles", r"C:\Program Files")
        )
        r_root = program_files / "R"

        if r_root.exists():
            for r_dir in sorted(r_root.glob("R-*"), reverse=True):
                candidates.extend(
                    [
                        r_dir / "bin" / "Rscript.exe",
                        r_dir / "bin" / "x64" / "Rscript.exe",
                    ]
                )

    seen = set()

    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except Exception:
            continue

        key = str(candidate).lower()

        if key in seen:
            continue

        seen.add(key)

        if candidate.exists():
            return candidate

    raise RuntimeError(
        "Rscript was not found. Set RSCRIPT_BIN to the full Rscript path."
    )


def sanitized_r_environment(rscript):
    env = os.environ.copy()

    for key in [
        "CONDA_PREFIX",
        "CONDA_DEFAULT_ENV",
        "CONDA_PROMPT_MODIFIER",
        "CONDA_EXE",
        "CONDA_PYTHON_EXE",
        "_CE_CONDA",
        "_CE_M",
        "PYTHONHOME",
        "PYTHONPATH",
        "R_HOME",
        "MKL_THREADING_LAYER",
        "MKL_SERVICE_FORCE_INTEL",
        "KMP_DUPLICATE_LIB_OK",
    ]:
        env.pop(key, None)

    path_parts = [
        part
        for part in env.get("PATH", "").split(os.pathsep)
        if part
    ]

    blocked_terms = [
        r"\.conda\envs",
        r"\anaconda",
        r"\miniconda",
        r"\conda",
    ]

    clean_parts = []

    for part in path_parts:
        normalized = part.lower().replace("/", "\\")

        if any(
            blocked.lower() in normalized
            for blocked in blocked_terms
        ):
            continue

        clean_parts.append(part)

    preferred = [
        str(Path(rscript).parent),
    ]

    if os.name == "nt":
        preferred.extend(
            [
                r"C:\Windows\System32",
                r"C:\Windows",
            ]
        )

    final_parts = []
    seen = set()

    for part in preferred + clean_parts:
        key = part.lower()

        if key in seen:
            continue

        seen.add(key)
        final_parts.append(part)

    env["PATH"] = os.pathsep.join(final_parts)

    if os.name == "nt":
        match = re.search(
            r"R-(\d+)\.(\d+)",
            str(rscript),
            flags=re.IGNORECASE,
        )

        if match and env.get("LOCALAPPDATA"):
            r_minor = f"{match.group(1)}.{match.group(2)}"

            user_lib = (
                Path(env["LOCALAPPDATA"])
                / "R"
                / "win-library"
                / r_minor
            )

            if user_lib.exists():
                env["R_LIBS_USER"] = str(user_lib)

    return env


def bridge_test(rscript, env):
    r_code = (
        'cat("R_VERSION=",R.version.string,"\\n",sep="");'
        'cat("MATRIX_INSTALLED=",'
        'requireNamespace("Matrix",quietly=TRUE),"\\n",sep="");'
        'cat("COPYKAT_INSTALLED=",'
        'requireNamespace("copykat",quietly=TRUE),"\\n",sep="");'
        'if(requireNamespace("copykat",quietly=TRUE))'
        'cat("COPYKAT_VERSION=",'
        'as.character(packageVersion("copykat")),"\\n",sep="")'
    )

    result = subprocess.run(
        [
            str(rscript),
            "--vanilla",
            "-e",
            r_code,
        ],
        env=env,
        capture_output=True,
        text=True,
        timeout=120,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(
            "Clean-environment R bridge test failed.\n"
            f"return_code={result.returncode}\n"
            f"stdout={result.stdout}\n"
            f"stderr={result.stderr}"
        )

    if "COPYKAT_INSTALLED=TRUE" not in result.stdout:
        raise RuntimeError(
            "Rscript works, but CopyKAT is not visible in the sanitized environment.\n"
            + result.stdout
        )

    print(result.stdout.strip())


rscript = find_rscript()
r_env = sanitized_r_environment(rscript)

print("[START] CopyKAT microbatch runner")
print("Rscript:", rscript)
print("Manifest:", MANIFEST)
print("R_LIBS_USER:", r_env.get("R_LIBS_USER", "<R default>"))

print("[BRIDGE TEST]")
bridge_test(rscript, r_env)

with MANIFEST.open(
    "r",
    encoding="utf-8",
    newline="",
) as handle:

    reader = csv.DictReader(
        handle,
        delimiter="\t",
    )

    for row in reader:

        batch_id = row["batch_id"]
        batch_dir = Path(row["batch_dir"])
        prediction = Path(row["expected_prediction"])
        r_script = Path(row["r_script"])

        print("-" * 60)
        print("[BATCH]", batch_id)

        if prediction.exists():
            print("[SKIP]", prediction)
            continue

        log = batch_dir / f"{batch_id}_run.log"

        batch_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        with log.open(
            "w",
            encoding="utf-8",
            errors="replace",
        ) as log_handle:

            try:
                proc = subprocess.run(
                    [
                        str(rscript),
                        "--vanilla",
                        str(r_script),
                    ],
                    cwd=str(batch_dir),
                    env=r_env,
                    stdout=log_handle,
                    stderr=subprocess.STDOUT,
                    timeout=TIMEOUT_SECONDS,
                    check=False,
                )

                status = int(proc.returncode)

            except subprocess.TimeoutExpired:
                status = 124

                log_handle.write(
                    f"\n[TIMEOUT] batch exceeded "
                    f"{TIMEOUT_SECONDS / 3600:.1f} hours.\n"
                )

            except Exception as error:
                status = 125

                log_handle.write(
                    "\n[RUNNER ERROR] "
                    + repr(error)
                    + "\n"
                )

        print("[STATUS]", batch_id, status)

        if status != 0:
            print("[STOP] Batch failed. Inspect:")
            print(log)
            sys.exit(status)

        if not prediction.exists():
            print("[STOP] Batch returned 0 but prediction is missing:")
            print(prediction)
            sys.exit(126)

print("[DONE] CopyKAT microbatch runner")
'''

runner = (
    runner.replace("__BASE_DIR__", str(OUT_BASE_DIR))
    .replace("__MANIFEST__", str(INPUT_MANIFEST_TSV))
    .replace("__TIMEOUT_SECONDS__", str(BATCH_TIMEOUT_HOURS * 3600))
)
MASTER_PYTHON.write_text(runner, encoding="utf-8")

MASTER_PS1.write_text(
    '$ErrorActionPreference = "Stop"\n'
    f'python "{MASTER_PYTHON}"\n'
    'exit $LASTEXITCODE\n',
    encoding="utf-8",
)

# Preserve the archived-style shell filename too.
shell = (
    '#!/usr/bin/env bash\n'
    'set -u\n'
    f'python "{MASTER_PYTHON.as_posix()}"\n'
)
MASTER_SH.write_text(shell, encoding="utf-8")

try:
    os.chmod(MASTER_SH, 0o755)
    os.chmod(MASTER_PYTHON, 0o755)
except Exception:
    pass

prepare_manifest = pd.DataFrame([{
    "base_dir": str(OUT_BASE_DIR),
    "master_shell": str(MASTER_SH),
    "manifest": str(INPUT_MANIFEST_TSV),
    "n_batches": len(batch_plan),
    "target_chunk_size": TARGET_CHUNK_SIZE,
    "ref_anchor_size": REF_ANCHOR_SIZE,
}])
if prepare_manifest.shape != (1, 6):
    raise AssertionError("Historical prepare-output manifest structure changed.")
write_tsv_replace(prepare_manifest, PREPARE_MANIFEST_TSV)

repro_audit = pd.DataFrame([
    {"metric": "source_h5ad", "value": str(SOURCE_H5AD)},
    {"metric": "source_shape", "value": f"{EXPECTED_N_CELLS}x{EXPECTED_USABLE_GENES}"},
    {"metric": "source_obs_names_sha256", "value": hash_strings(obs.index.astype(str))},
    {"metric": "source_gene_order_sha256", "value": hash_strings(source_genes)},
    {"metric": "copykat_gene_order_sha256", "value": hash_strings(usable_genes)},
    {"metric": "source_gene_order_already_historical", "value": str(gene_order_identity)},
    {"metric": "reference_pool_cells", "value": str(len(ref_pool))},
    {"metric": "reference_anchor_cells", "value": str(len(ref_anchor_indices))},
    {"metric": "reference_anchor_cell_ids_sha256", "value": hash_strings(obs.index[ref_anchor_indices].astype(str))},
    {"metric": "target_cells_total", "value": str(int(batch_plan["n_target_cells"].sum()))},
    {"metric": "n_batches", "value": str(len(batch_plan))},
    {"metric": "historical_anchor_identity_claim", "value": "not asserted: canonical row order may differ from archived pre-renaming object"},
])
write_tsv_replace(repro_audit, REPRO_AUDIT_TSV)

display(prepare_manifest)
display(repro_audit)

rscript_detected = os.environ.get("RSCRIPT_BIN") or shutil.which("Rscript")

print("\nDONE — canonical historical CopyKAT micro-batch preparation reproduced with memory-safe execution.")
print("Gene-position report:", gene_report.shape)
print("Batch plan:", batch_plan.shape)
print("Input manifest:", manifest.shape)
print("Prepare output manifest:", prepare_manifest.shape)
print("n batches:", len(batch_plan))
print("target cells covered:", int(batch_plan["n_target_cells"].sum()))
print("reference pool:", len(ref_pool))
print("reference anchors:", len(ref_anchor_indices))
print("usable genes:", EXPECTED_USABLE_GENES)
print("Rscript detected:", rscript_detected)
print("\nRecommended Windows runner:")
print(MASTER_PYTHON)
print("\nPowerShell wrapper:")
print(MASTER_PS1)
print("\nScratch:")
print(SCRATCH_ROOT)

## Expected successful completion

```text
Gene-position report:       (1, 8)
Batch plan:              (176, 10)
Input manifest:          (176, 14)
Prepare output manifest:     (1, 6)

n batches:                  176
target cells covered:    351427
reference pool:            17449
reference anchors:          1000
usable genes:              15163
```

Each batch contains the compressed raw-count matrix, gene list, barcode list,
cell metadata, and an R script. The first 175 target chunks contain 2,000 target
cells; the final target chunk contains 1,427.

On Windows, after this notebook finishes successfully, run the generated
cross-platform Python runner. It executes CopyKAT sequentially, skips existing
prediction TSVs, and enforces the historical 8-hour timeout per batch.